In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

from sklearn.ensemble import VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [3]:
data_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis/")

In [4]:
grouped_train_logs = pd.read_csv(data_path / 'grouped_train_logs_advanced.csv').drop(columns={'Unnamed: 0'})
X_train_scores = pd.read_csv(data_path / 'X_train_scores.csv').drop(columns={'Unnamed: 0'})
X_test_scores = pd.read_csv(data_path / 'X_test_scores.csv').drop(columns={'Unnamed: 0'})

In [5]:
cols_to_drop = ['id', 'max_up_time', 'min_down_time', 'activity_len_per_essay_Nonproduction']

In [6]:
# Splitting the grouped_train_logs into train and test sets based on IDs
train_data = grouped_train_logs.merge(X_train_scores, on='id').drop(columns=cols_to_drop)
test_data = grouped_train_logs.merge(X_test_scores, on='id').drop(columns=cols_to_drop)

In [7]:
# Separating features and target variable
X_train = train_data.drop(['score'], axis=1)
y_train = train_data['score']
X_test = test_data.drop(['score'], axis=1)
y_test = test_data['score']

In [13]:
def train_and_evaluate(model, X_train=X_train, X_test=X_test):
    """
    Trenuje podany model i oblicza RMSE na zbiorach treningowym i testowym
    """
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    return rmse_train, rmse_test

In [14]:
scaler = StandardScaler()

In [15]:
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

## Model Ensembling

In [16]:
linear_reg = LinearRegression()
nn_reg = MLPRegressor(max_iter=1000, early_stopping=True, random_state=1)
rf_reg = RandomForestRegressor(max_depth=4, random_state=1)

In [17]:
estimators = [
    ('linear', linear_reg),
    ('nn', nn_reg),
    ('rf', rf_reg)
]

In [18]:
res = {}

In [21]:
for n, e in estimators:
    results = train_and_evaluate(e, X_train_normalized, X_test_normalized)
    res[n] = results
    print(n, results[0], results[1])

linear 0.6731355546576886 0.7170823262426633
nn 0.5737502973282158 0.7198432054373123
rf 0.6434572836175315 0.699241356683026


In [22]:
ereg = VotingRegressor(estimators=estimators)

In [23]:
train_and_evaluate(ereg, X_train_normalized, X_test_normalized)

(np.float64(0.6054131561548046), np.float64(0.6846519125905873))